# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample/**/*.parquet')",
}

HF_TOKEN: ··········


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [6]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224


In [7]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609,2.0,0.037221,0.823821,34.0,56.0,0.607143
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667,1.0,0.076923,0.446154,93.0,93.0,1.000000
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574,13.0,0.014321,0.718867,226.0,857.0,0.263711
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051,2.0,0.129032,0.760753,22.0,41.0,0.536585
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224,1.0,0.053045,0.843811,105.0,105.0,1.000000


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
The unit of analysis is one content page (webpage) over a reporting period. I will use a mid-panel time window when exploring the data and avoid using the final month for feature design to reduce leakage risk.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) AS rows
FROM {TABLES['fact_daily_sample']}
""").df()

,rows
0,11694072


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features
- gsc_impressions
- gsc_clicks
- average_position
- ctr
- visible_queries

## Label
is_declining (or the proxy defined later)

## Context
client_hash_id
content_hash_id
report_date

## Excluded
Future information and any label-derived columns because they would leak the answer into the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
COUNT(*) AS records
FROM {TABLES['fact_daily_sample']}
GROUP BY 1,2
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,records
0,client_3ffa76342f366962,content_1a6296faee432dae,30
1,client_3ffa76342f366962,content_dc34c661d63e55a9,30
2,client_3ffa76342f366962,content_dd83cb75985afc9c,30
3,client_3ffa76342f366962,content_42e4dc3c4026a190,30
4,client_3ffa76342f366962,content_d86e1c84849226ba,30
5,client_3ffa76342f366962,content_bcfcf9f17422d40f,30
6,client_3ffa76342f366962,content_039395070f4f08d1,30
7,client_3ffa76342f366962,content_68d39d68301bd40f,30
8,client_3ffa76342f366962,content_6ca55072a1e42f11,30
9,client_3ffa76342f366962,content_df2adaab19a6ef53,30


In [10]:
con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date,
COUNT(*) AS total_rows
FROM {TABLES['fact_daily_sample']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-06-01,2026-06-30,11694072


In [11]:
con.sql(f"""
SELECT
COUNT(*) AS total_rows,
COUNT(gsc_impressions) AS impressions_present,
COUNT(gsc_clicks) AS clicks_present
FROM {TABLES['fact_daily_sample']}
""").df()

,total_rows,impressions_present,clicks_present
0,11694072,11694072,11694072


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This dataset cannot capture every reason why a page should be refreshed. It does not directly measure content quality, business priorities, or user intent. The data should therefore be used for decision support rather than as the sole basis for content refresh decisions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.